# All-latent capital-relation localisation

This is an independent robustness test of the 10,000-prompt capitals TopK-128 SAE. It does **not** retrain the model or SAE and does not use an attribution graph to define candidate features.

All 57,344 latents across layers 4, 8, 12, 16, 20, 24 and 28 are ranked using paired capital and inverse-country prompts on discovery countries. The feature order is then frozen before activation and causal confirmation on disjoint countries. Every country used in the earlier graph-seeded screen is excluded.

The predeclared primary panel is Top-3, matching the compact graph result. The run requests 24 discovery and 24 confirmation countries. If fewer than 48 baseline-correct unused countries exist, a baseline-only rule permits equal splits down to 16+16; feature activations cannot affect that fallback.

## 1. Mount Drive and pull the repository

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import subprocess

repo_url = 'https://github.com/evey-dev/test_run.git'
repo_dir = '/content/test_run'

def run_cmd(command):
    print('$', ' '.join(map(str, command)), flush=True)
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)

github_ok = False
try:
    checkout = repo_dir
    if os.path.isdir(os.path.join(checkout, '.git')):
        run_cmd(['git', '-C', checkout, 'pull', '--ff-only'])
    else:
        if os.path.exists(checkout) and os.listdir(checkout):
            checkout = '/content/test_run_github'
        if os.path.isdir(os.path.join(checkout, '.git')):
            run_cmd(['git', '-C', checkout, 'pull', '--ff-only'])
        elif os.path.exists(checkout) and os.listdir(checkout):
            raise RuntimeError(f'{checkout} exists but is not a git repository')
        else:
            run_cmd(['git', 'clone', '--depth', '1', repo_url, checkout])
    os.chdir(checkout)
    github_ok = True
    print('Using GitHub checkout:', os.getcwd())
except Exception as exc:
    print('GitHub checkout failed; using Drive project.zip backup.')
    print(repr(exc))

if not github_ok:
    zip_path = '/content/drive/MyDrive/mphil-project/project.zip'
    if not os.path.exists(zip_path):
        raise FileNotFoundError(f'Could not find {zip_path}')
    run_cmd(['unzip', '-q', '-o', zip_path, '-d', '/content/'])
    for candidate in ['/content/test_run', '/content/mphil_project/test_run', '/content']:
        if os.path.isdir(os.path.join(candidate, 'src')) and os.path.isdir(os.path.join(candidate, 'configs')):
            os.chdir(candidate)
            break
    else:
        raise FileNotFoundError('Could not locate the extracted repository root')

print('Current working directory:', os.getcwd())

## 2. Install dependencies and regenerate deterministic data files

In [ ]:
!pip install -q --upgrade "transformers>=4.51.0" accelerate
!pip install -q -e .
!python data/generate_datasets.py --capitals
!python data/generate_large_capitals_dataset.py --count 10000 --seed 787 --base-csv data/capitals_data.csv --output data/capitals_large_10000.csv

from pathlib import Path
required = [
    Path('src/capitals_relation_balanced_localization.py'),
    Path('configs/sae_capitals_large_10000_topk128_config.yaml'),
    Path('data/capitals_data.csv'),
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('Push the new capitals follow-up files to GitHub: ' + ', '.join(missing))

## 3. Restore the existing 10k SAE and define persistent outputs

In [ ]:
import json
import shutil
import sys

LAYERS = [4, 8, 12, 16, 20, 24, 28]
DRIVE_ROOT = Path('/content/drive/MyDrive/mphil-project')
DRIVE_CHECKPOINTS = DRIVE_ROOT / 'mechanistic_data' / 'capitals_large_data_test' / 'checkpoints_topk128'
DRIVE_GRAPH_OUTPUT = DRIVE_ROOT / 'outputs' / 'capitals_large_data_test'
DRIVE_OUTPUT = DRIVE_ROOT / 'outputs' / 'capitals_balanced_localization'
LOCAL_CHECKPOINTS = Path('mechanistic_data/sae_checkpoints_capitals_large_10000_topk128')
LOCAL_OUTPUT = Path('outputs/capitals_balanced_localization')

for path in (DRIVE_OUTPUT, LOCAL_CHECKPOINTS, LOCAL_OUTPUT):
    path.mkdir(parents=True, exist_ok=True)

for layer in LAYERS:
    for name in (f'sae_layer{layer}.pt', f'sae_layer{layer}_metadata.json'):
        source = DRIVE_CHECKPOINTS / name
        destination = LOCAL_CHECKPOINTS / name
        if not source.exists():
            raise FileNotFoundError(f'Missing 10k capitals SAE artifact: {source}')
        if not destination.exists():
            shutil.copy2(source, destination)
            print('Restored', destination)

PRIOR_GRAPH_RESULT = DRIVE_GRAPH_OUTPUT / 'capitals_large_10000_topk128_relation_screen.json'
if not PRIOR_GRAPH_RESULT.exists():
    raise FileNotFoundError(
        f'The prior graph result is required to exclude its countries: {PRIOR_GRAPH_RESULT}'
    )

RESULT_PATH = DRIVE_OUTPUT / 'capitals_10k_balanced_relation_localization.json'
CACHE_PATH = DRIVE_OUTPUT / 'capitals_10k_balanced_relation_activations.npz'
print('Prior graph result:', PRIOR_GRAPH_RESULT)
print('Balanced result:', RESULT_PATH)
print('Activation cache:', CACHE_PATH)

## 4. Run all-latent discovery and frozen confirmation

The result is checkpointed after every causal panel and the activation matrix is cached. Rerunning this cell resumes safely. Do not add `--overwrite` unless intentionally restarting the complete protocol.

In [ ]:
command = [
    sys.executable, '-u', '-m', 'src.capitals_relation_balanced_localization',
    '--sae-config', 'configs/sae_capitals_large_10000_topk128_config.yaml',
    '--capitals-csv', 'data/capitals_data.csv',
    '--sae-corpus-csv', 'data/capitals_large_10000.csv',
    '--exclude-json', str(PRIOR_GRAPH_RESULT),
    '--exclude-country', 'Jordan',
    '--desired-cases-per-split', '24',
    '--minimum-cases-per-split', '16',
    '--cities-per-country', '6',
    '--seed', '8787',
    '--candidate-batch-size', '16',
    '--activation-batch-size', '8',
    '--panel-sizes', '1', '3', '5', '10', '20',
    '--primary-panel-size', '3',
    '--random-panels', '5',
    '--minimum-active-fraction', '0.10',
    '--minimum-positive-pair-fraction', '0.60',
    '--output', str(RESULT_PATH),
    '--activation-cache', str(CACHE_PATH),
]
run_cmd(command)

## 5. Inspect the predeclared result

In [ ]:
import pandas as pd
from IPython.display import display

result = json.loads(RESULT_PATH.read_text())
primary = result['primary_result']
print('Status:', result['status'])
print('Excluded previously intervened countries:', result['exclusions']['excluded_country_count'])
print('Discovery countries:', len(result['split']['discovery_countries']))
print('Confirmation countries:', len(result['split']['confirmation_countries']))
print('Baseline-only fallback used:', result['split']['rule']['fallback_used'])
print('All-latent candidates after fixed filters:', result['sae_feature_discovery']['candidate_count_after_fixed_filters'])
print('Predeclared Top-3 passed:', primary['supports_compact_capital_relation_selectivity_under_predeclared_rule'])
print(primary['interpretation_gate'])

ranking = pd.DataFrame(result['sae_feature_discovery']['top_200_ranking'][:20])
display(ranking[[
    'rank', 'key', 'paired_standardised_effect', 'positive_discovery_pairs',
    'total_discovery_pairs', 'capital_active_fraction',
    'inverse_country_active_fraction',
]])

panel_rows = []
for panel in result['causal_confirmation']['panels']:
    activation = panel['activation_validation']['confirmation']
    causal = panel['causal_summary']
    panel_rows.append({
        'panel': panel['name'],
        'kind': panel['kind'],
        'features': panel['feature_count'],
        'activation effect': activation['mean_capital_minus_inverse_score'],
        'activation CI': activation['bootstrap_95_ci_mean_capital_minus_inverse_score'],
        'capital delta': causal['mean_capital_prompt_delta'],
        'inverse control delta': causal['mean_inverse_country_prompt_delta'],
        'causal specificity': causal['mean_relation_specific_difference'],
        'causal CI': causal['bootstrap_95_ci_mean_relation_specific_difference'],
    })
display(pd.DataFrame(panel_rows))

## 6. Compare the independent Top-3 with the graph-seeded Top-3

In [ ]:
graph_result = json.loads(PRIOR_GRAPH_RESULT.read_text())
graph_panel = next(
    panel for panel in graph_result['confirmation']['panels']
    if panel['name'] == 'top_3_primary'
)
balanced_panel = next(
    panel for panel in result['causal_confirmation']['panels']
    if panel['name'] == 'top_3_primary'
)
graph_ids = {feature['key'] for feature in graph_panel['features']}
balanced_ids = {feature['key'] for feature in balanced_panel['features']}
print('Graph Top-3:', sorted(graph_ids))
print('Balanced Top-3:', sorted(balanced_ids))
print('Overlapping IDs:', sorted(graph_ids & balanced_ids))
print('Graph held-out causal specificity:', graph_panel['summary']['mean_relation_specific_difference'])
print('Balanced held-out causal specificity:', balanced_panel['causal_summary']['mean_relation_specific_difference'])
print('These values use different held-out countries and are not a paired method comparison.')

## 7. Save a compact diagnostic figure and local copies

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))

top = result['sae_feature_discovery']['top_200_ranking'][:10]
axes[0].barh(
    [row['key'] for row in reversed(top)],
    [row['paired_standardised_effect'] for row in reversed(top)],
    color='#7B5AA6',
)
axes[0].set(title='Discovery ranking', xlabel='Paired standardised effect')

ranked_panels = [
    panel for panel in result['causal_confirmation']['panels']
    if panel['kind'] == 'balanced_activation_ranked_prefix'
]
sizes = [panel['feature_count'] for panel in ranked_panels]
activation_means = [panel['activation_validation']['confirmation']['mean_capital_minus_inverse_score'] for panel in ranked_panels]
axes[1].plot(sizes, activation_means, marker='o', color='#168C87')
axes[1].axhline(0, color='0.4', linewidth=1)
axes[1].set(title='Held-out activation', xlabel='Top-N features', ylabel='Capital minus inverse score')

causal_means = [panel['causal_summary']['mean_relation_specific_difference'] for panel in ranked_panels]
causal_lo = [panel['causal_summary']['bootstrap_95_ci_mean_relation_specific_difference'][0] for panel in ranked_panels]
causal_hi = [panel['causal_summary']['bootstrap_95_ci_mean_relation_specific_difference'][1] for panel in ranked_panels]
axes[2].errorbar(
    sizes, causal_means,
    yerr=[np.asarray(causal_means) - np.asarray(causal_lo), np.asarray(causal_hi) - np.asarray(causal_means)],
    marker='o', color='#C94F45', capsize=3,
)
axes[2].axhline(0, color='0.4', linewidth=1)
axes[2].set(title='Held-out causal specificity', xlabel='Top-N features', ylabel='Capital minus inverse gap delta')

fig.suptitle('All-latent capital-relation localisation', fontweight='bold')
fig.tight_layout()
FIGURE_PNG = DRIVE_OUTPUT / 'fig_capitals_balanced_relation_localization.png'
FIGURE_PDF = DRIVE_OUTPUT / 'fig_capitals_balanced_relation_localization.pdf'
fig.savefig(FIGURE_PNG, dpi=220, bbox_inches='tight')
try:
    fig.savefig(FIGURE_PDF, bbox_inches='tight')
except ImportError as exc:
    if 'FontPath' not in str(exc):
        raise
    from PIL import Image as PILImage
    with PILImage.open(FIGURE_PNG) as rendered:
        rendered.convert('RGB').save(FIGURE_PDF, 'PDF', resolution=220.0)
plt.show()

for source in (RESULT_PATH, FIGURE_PNG, FIGURE_PDF):
    destination = LOCAL_OUTPUT / source.name
    shutil.copy2(source, destination)
    print('Copied', destination)

## Interpretation and stopping rule

- **Primary Top-3 passes:** report independent all-latent confirmation of a compact capital-relation panel. Agreement in feature IDs with the graph panel is convergence; different IDs with the same held-out causal effect indicate a distributed or non-unique SAE basis.
- **Activation passes but causal specificity fails:** the relation is decodable from the selected coordinates, but they are not established as a selective causal mechanism.
- **Primary Top-3 fails while a larger panel succeeds:** treat the larger result as exploratory only. Do not replace the primary claim without a new frozen replication.
- **Primary and secondary panels fail:** retain the successful graph-seeded result and report that exhaustive activation ranking did not independently recover a compact panel.
- Stop after this run; do not tune filters or replace the primary panel using confirmation results.